# 🎓 GradeForge — RAG Pipeline

**End-to-end pipeline:** MinerU Markdown → Structure Parser → Document Classifier → Chunking → Metadata → Embeddings → Pinecone → Retrieval → Answer Review Agent → Score + Feedback

---

## 1. Setup & Imports

In [4]:
import os
import re
import json
import uuid
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from groq import Groq
from ollama import chat as ollama_chat

# Load environment variables
load_dotenv(Path("../../.env"))

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_LLM_MODEL = os.getenv("GROQ_LLM_MODEL")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_LLM_MODEL = os.getenv("GEMINI_LLM_MODEL")
RERANK_MODEL = os.getenv("RERANK_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ PINECONE_API_KEY loaded:", bool(PINECONE_API_KEY))
print("✅ GROQ_API_KEY loaded:", bool(GROQ_API_KEY))
print("✅ GROQ_LLM_MODEL loaded:", bool(GROQ_LLM_MODEL))
print("✅ GEMINI_API_KEY loaded:", bool(GEMINI_API_KEY))

print("✅ GEMINI_LLM_MODEL loaded:", bool(GEMINI_LLM_MODEL))

# ------------------------------------------------------------
# Ollama configuration
# ------------------------------------------------------------
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:4b")

print("✅ Ollama model:", OLLAMA_MODEL)
print("✅ Rerank model:", RERANK_MODEL)

✅ PINECONE_API_KEY loaded: True
✅ GROQ_API_KEY loaded: True
✅ GROQ_LLM_MODEL loaded: True
✅ GEMINI_API_KEY loaded: True
✅ GEMINI_LLM_MODEL loaded: True
✅ Ollama model: qwen3:4b
✅ Rerank model: cross-encoder/ms-marco-MiniLM-L-6-v2


## 2. Configuration

In [5]:
# --- Configuration ---

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # 384-dimensional, fast & good quality
EMBEDDING_DIM = 384

# Pinecone
PINECONE_INDEX_NAME = "gradeforge-rag"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

# Pinecone namespaces (logical separation per architecture doc)
NS_BOOK = "book_chunks"
NS_ANSWER_KEY = "answer_key_chunks"
NS_STUDENT = "student_answer_chunks"

# Chunking
BOOK_CHUNK_MAX_TOKENS = 800
BOOK_CHUNK_MIN_TOKENS = 100
BOOK_CHUNK_OVERLAP_TOKENS = 50

# Groq LLM
GROQ_MODEL = GROQ_LLM_MODEL

# Ollama LLM — PRIMARY
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:4b")

# Provider selection
PRIMARY_LLM = "gemini"

# Whether Groq should be used automatically if Ollama fails
ENABLE_GROQ_FALLBACK = True
ENABLE_OLLAMA_FALLBACK = False

# Retrieval
TOP_K = 5

# Guardrail & Relevance Thresholds
# Discard chunks with cosine similarity below this score to avoid out-of-domain hallucinations
SIMILARITY_THRESHOLD = 0.30

print("✅ Configuration set")
print(f"🤖 Primary LLM: → {PRIMARY_LLM}")
print(f"🔄 Groq fallback: {ENABLE_GROQ_FALLBACK} → {GROQ_MODEL}")
print(f"🛡️ Relevance similarity threshold: {SIMILARITY_THRESHOLD}")

✅ Configuration set
🤖 Primary LLM: → gemini
🔄 Groq fallback: True → openai/gpt-oss-120b
🛡️ Relevance similarity threshold: 0.3


## 3. Data Models — Chunk Schema

In [6]:
@dataclass
class Chunk:
    """Canonical chunk schema as defined in the architecture document."""
    chunk_id: str
    parent_id: Optional[str]
    document_id: str
    document_type: str          # "book", "answer_key", "student_paper"
    chunk_type: str             # "concept", "definition", "formula", "example", "figure", "table", "question", "answer", "qa", "student_answer"
    content: str
    chapter: Optional[str] = None
    section: Optional[str] = None
    subsection: Optional[str] = None
    question_id: Optional[str] = None
    page_start: Optional[int] = None
    page_end: Optional[int] = None
    source: Optional[str] = None
    language: Optional[str] = "en"
    subject: Optional[str] = None
    formula_reference: Optional[str] = None
    image_reference: Optional[str] = None
    table_reference: Optional[str] = None

    def to_metadata(self) -> dict:
        """Convert to flat metadata dict for Pinecone (no None values)."""
        d = asdict(self)
        # Remove content (stored separately) and None values
        d.pop("content", None)
        return {k: v for k, v in d.items() if v is not None}


def generate_chunk_id(content: str, doc_id: str, index: int) -> str:
    """Generate a deterministic chunk ID."""
    raw = f"{doc_id}::{index}::{content[:100]}"
    return hashlib.md5(raw.encode()).hexdigest()


print("✅ Chunk schema defined")

✅ Chunk schema defined


## 4. Markdown Structure Parser

Parses MinerU markdown output into structured blocks with heading hierarchy.

In [7]:
@dataclass
class MarkdownBlock:
    """A parsed block from the markdown file."""
    heading_level: int            # 0 = body text, 1 = h1, 2 = h2, etc.
    heading_text: str             # The heading text (empty for body)
    content: str                  # The content under this heading
    hierarchy: list               # [h1_text, h2_text, h3_text, ...]
    has_formula: bool = False
    has_table: bool = False
    has_list: bool = False


def parse_markdown(md_text: str) -> list[MarkdownBlock]:
    """Parse markdown into structured blocks preserving heading hierarchy."""
    lines = md_text.split("\n")
    blocks = []
    
    # Track current heading hierarchy
    current_hierarchy = {}  # level -> heading text
    current_heading_level = 0
    current_heading_text = ""
    current_content_lines = []
    
    heading_pattern = re.compile(r"^(#{1,6})\s+(.+)$")
    formula_pattern = re.compile(r"\$\$|\$[^$]+\$")
    table_pattern = re.compile(r"^\|.*\|$")
    list_pattern = re.compile(r"^\s*[•\-\*]\s+|^\s*\d+[\.\)]\s+")
    
    def flush_block():
        """Save the current block."""
        content = "\n".join(current_content_lines).strip()
        if content or current_heading_text:
            # Build hierarchy list
            hierarchy = []
            for lvl in sorted(current_hierarchy.keys()):
                if lvl <= current_heading_level:
                    hierarchy.append(current_hierarchy[lvl])
            
            has_formula = bool(formula_pattern.search(content))
            has_table = bool(table_pattern.search(content))
            has_list = bool(list_pattern.search(content))
            
            blocks.append(MarkdownBlock(
                heading_level=current_heading_level,
                heading_text=current_heading_text,
                content=content,
                hierarchy=hierarchy.copy(),
                has_formula=has_formula,
                has_table=has_table,
                has_list=has_list,
            ))
    
    for line in lines:
        match = heading_pattern.match(line.strip())
        if match:
            # Flush previous block
            flush_block()
            
            level = len(match.group(1))
            heading = match.group(2).strip()
            
            # Update hierarchy: set current level, clear deeper levels
            current_hierarchy[level] = heading
            for lvl in list(current_hierarchy.keys()):
                if lvl > level:
                    del current_hierarchy[lvl]
            
            current_heading_level = level
            current_heading_text = heading
            current_content_lines = []
        else:
            current_content_lines.append(line)
    
    # Flush the last block
    flush_block()
    
    return blocks


# --- Test with sample ---
sample_md = """# Chapter 1: Intro
## 1.1 Overview
This is the overview.
## 1.2 Details
### Subsection A
Detail content here.
$$E = mc^2$$
"""

test_blocks = parse_markdown(sample_md)
for b in test_blocks:
    print(f"  H{b.heading_level}: {b.heading_text} | hierarchy={b.hierarchy} | formula={b.has_formula}")
    print(f"    content: {b.content[:80]}...")
    print()

  H1: Chapter 1: Intro | hierarchy=['Chapter 1: Intro'] | formula=False
    content: ...

  H2: 1.1 Overview | hierarchy=['Chapter 1: Intro', '1.1 Overview'] | formula=False
    content: This is the overview....

  H2: 1.2 Details | hierarchy=['Chapter 1: Intro', '1.2 Details'] | formula=False
    content: ...

  H3: Subsection A | hierarchy=['Chapter 1: Intro', '1.2 Details', 'Subsection A'] | formula=True
    content: Detail content here.
$$E = mc^2$$...



## 5. Document Classifier

Classifies a markdown document as `book`, `answer_key`, or `student_paper` using heuristic signals.

In [8]:
def classify_document(md_text: str, filename: str = "") -> str:
    """
    Classify a document as 'book', 'answer_key', or 'student_paper'.
    
    Uses multiple signals:
    - Filename hints
    - Q&A pattern density
    - Heading structure (chapters vs questions)
    - Answer markers
    - Content length and structure
    """
    text_lower = md_text.lower()
    fname_lower = filename.lower()
    
    # --- Signal: Filename ---
    if any(kw in fname_lower for kw in ["book", "textbook", "material", "study", "chapter"]):
        return "book"
    if any(kw in fname_lower for kw in ["answer_key", "answerkey", "key", "solution", "correct","QA_sheet","QA-sheet"]):
        return "answer_key"
    if any(kw in fname_lower for kw in ["student", "submission", "response", "paper", "exam","test"]):
        return "student_paper"
    
    # --- Signal: Q&A pattern density ---
    qa_patterns = [
        r"(?:^|\n)\s*(?:Q|Question|Que)\.?\s*\d+",
        r"(?:^|\n)\s*\d+[\.\)\]]\s",
        r"(?:^|\n)\s*\(\d+\)",
    ]
    qa_count = sum(len(re.findall(p, md_text, re.IGNORECASE)) for p in qa_patterns)
    
    answer_markers = len(re.findall(
        r"(?:Answer|Ans|Solution|Sol)\s*[:：]", md_text, re.IGNORECASE
    ))
    
    # --- Signal: Chapter/heading structure ---
    chapter_pattern = len(re.findall(r"^#{1,2}\s+.*(?:chapter|unit|module)", md_text, re.IGNORECASE | re.MULTILINE))
    
    # --- Signal: Heading-based question detection ---
    heading_questions = len(re.findall(r"^#{1,3}\s+.*(?:question|Q)\s*\d+", md_text, re.IGNORECASE | re.MULTILINE))
    
    # --- Decision ---
    # If many chapters and few Q&A patterns → book
    if chapter_pattern >= 2 and qa_count < 5:
        return "book"
    
    # If many Q&A with answer markers → answer_key
    if (qa_count >= 3 or heading_questions >= 3) and answer_markers >= 2:
        return "answer_key"
    
    # If Q&A but few/no answer markers → student_paper
    if qa_count >= 3 or heading_questions >= 3:
        # Check if answers are present alongside questions
        if answer_markers >= qa_count * 0.5:
            return "answer_key"
        return "student_paper"
    
    # Default: if it has lots of content with headings → book
    if len(md_text) > 5000:
        return "book"
    
    return "book"  # safe default


# --- Test ---
print("Classification tests:")
print(f"  'maths-ch-8-book.pdf' → {classify_document('# Chapter 1...', 'maths-ch-8-book.pdf')}")
print(f"  'mathsch-8-QA-sheet.pdf'   → {classify_document('## Question 1\nAnswer: ...\n## Question 2\nAnswer: ...\n## Question 3\nAnswer: ...', 'mathsch-8-QA-sheet.pdf')}")
print(f"  'student.pdf'  → {classify_document('Q1. What is AI?\nAI is...\nQ2. Explain ML\nML is...\nQ3. Define DL\nDL is...', 'student_test.pdf')}")

Classification tests:
  'maths-ch-8-book.pdf' → book
  'mathsch-8-QA-sheet.pdf'   → answer_key
  'student.pdf'  → student_paper


In [9]:
# ═══════════════════════════════════════════════════════════
# 6-PRE. JSON Page-Map Builder
# Build a mapping: text_snippet → page_number from MinerU JSON
# This allows us to attach page numbers to chunks
# ═══════════════════════════════════════════════════════════

def _extract_block_text(content_items) -> str:
    """Recursively extract plain text from MinerU JSON content array."""
    texts = []
    if isinstance(content_items, list):
        for item in content_items:
            if isinstance(item, dict):
                if 'content' in item:
                    if isinstance(item['content'], str):
                        texts.append(item['content'])
                    elif isinstance(item['content'], list):
                        texts.append(_extract_block_text(item['content']))
            elif isinstance(item, str):
                texts.append(item)
    return ' '.join(texts).strip()


def build_page_map(json_path: str) -> dict:
    """
    Parse MinerU JSON and build a mapping of text snippets to page numbers.
    
    Returns:
        dict: { 'text snippet (first 120 chars, lowered)': page_number (1-indexed) }
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    page_map = {}  # snippet → page_num
    
    for page in data['pages']:
        page_num = page['page_idx'] + 1  # Convert 0-indexed to 1-indexed
        
        for block in page['blocks']:
            # Skip non-content blocks
            if block['type'] in ('page_number', 'header', 'aside_text', 'footer'):
                continue
            
            text = _extract_block_text(block['content'])
            if text and len(text.strip()) > 20:
                # Use first 120 chars lowered as key for matching
                key = text.strip()[:120].lower()
                page_map[key] = page_num
    
    return page_map


def find_page_for_text(text: str, page_map: dict) -> int:
    """
    Find the page number for a chunk of text by matching against the page_map.
    Uses substring matching to handle text that spans blocks.
    
    Returns: page number (1-indexed) or None if not found.
    """
    if not page_map or not text:
        return None
    
    text_lower = text.strip().lower()
    
    # Strategy 1: Exact prefix match (first 120 chars)
    prefix_key = text_lower[:120]
    if prefix_key in page_map:
        return page_map[prefix_key]
    
    # Strategy 2: Find the best overlapping key
    best_page = None
    best_overlap = 0
    text_words = set(text_lower[:200].split())
    
    for key, page_num in page_map.items():
        key_words = set(key.split())
        overlap = len(text_words & key_words)
        if overlap > best_overlap and overlap >= 5:
            best_overlap = overlap
            best_page = page_num
    
    return best_page


print("✅ JSON page-map builder defined")

✅ JSON page-map builder defined


## 6. Chunking Strategies

Three document-type-specific chunkers as per the architecture:
1. **Book** → Heading-aware + semantic chunking
2. **Answer Key** → Question-aware hierarchical chunking
3. **Student Paper** → Question-aware hierarchical chunking

In [18]:
def _estimate_tokens(text: str) -> int:
    """Rough token count estimate (words * 1.3)."""
    return int(len(text.split()) * 1.3)


def _detect_chunk_type(content: str) -> str:
    """Detect the type of content in a chunk."""
    content_lower = content.lower()
    
    if re.search(r"\$\$|\\frac|\\sum|\\int", content):
        return "formula"
    if re.search(r"^\|.*\|$", content, re.MULTILINE):
        return "table"
    if re.search(r"!\[.*\]\(.*\)", content):
        return "figure"
    if any(kw in content_lower for kw in ["definition:", "is defined as", "refers to"]):
        return "definition"
    if any(kw in content_lower for kw in ["example:", "for example", "e.g.", "for instance"]):
        return "example"
    if any(kw in content_lower for kw in ["steps:", "procedure:", "algorithm:", "step 1"]):
        return "procedure"
    return "concept"


# ═══════════════════════════════════════════════════════════
# 6A. BOOK CHUNKER — Heading-aware + semantic chunking
#     NOW WITH PAGE NUMBER SUPPORT from JSON page-map
# ═══════════════════════════════════════════════════════════

def chunk_book(blocks: list[MarkdownBlock], doc_id: str, source: str = "", page_map: dict = None) -> list[Chunk]:
    """
    Heading-aware + hierarchy-preserving chunker for book/study material.
    
    - Preserves heading hierarchy in metadata
    - Splits large sections into ~300-800 token chunks
    - Keeps formulas with their context
    - NEW: Attaches page_start/page_end from JSON page-map
    """
    chunks = []
    chunk_idx = 0
    
    for block in blocks:
        if not block.content.strip():
            continue
        
        # Determine chapter/section/subsection from hierarchy
        chapter = block.hierarchy[0] if len(block.hierarchy) > 0 else None
        section = block.hierarchy[1] if len(block.hierarchy) > 1 else None
        subsection = block.hierarchy[2] if len(block.hierarchy) > 2 else None
        
        # Full content = heading + body
        full_content = block.content
        if block.heading_text:
            full_content = f"{block.heading_text}\n\n{block.content}"
        
        # NEW: Find page number from JSON page-map
        page_num = find_page_for_text(full_content, page_map) if page_map else None
        
        token_count = _estimate_tokens(full_content)
        
        if token_count <= BOOK_CHUNK_MAX_TOKENS:
            # Small enough — keep as single chunk
            chunk_type = _detect_chunk_type(full_content)
            if block.has_formula:
                chunk_type = "formula"
            
            chunks.append(Chunk(
                chunk_id=generate_chunk_id(full_content, doc_id, chunk_idx),
                parent_id=None,
                document_id=doc_id,
                document_type="book",
                chunk_type=chunk_type,
                content=full_content,
                chapter=chapter,
                section=section,
                subsection=subsection,
                page_start=page_num,
                page_end=page_num,
                source=source,
            ))
            chunk_idx += 1
        else:
            # Too large — split by paragraphs, then by size
            parent_id = generate_chunk_id(full_content, doc_id, chunk_idx)
            paragraphs = re.split(r"\n\n+", full_content)
            
            current_text = ""
            sub_idx = 0
            
            for para in paragraphs:
                if _estimate_tokens(current_text + "\n\n" + para) > BOOK_CHUNK_MAX_TOKENS and current_text:
                    chunk_type = _detect_chunk_type(current_text)
                    sub_page = find_page_for_text(current_text, page_map) if page_map else None
                    chunks.append(Chunk(
                        chunk_id=generate_chunk_id(current_text, doc_id, chunk_idx),
                        parent_id=parent_id,
                        document_id=doc_id,
                        document_type="book",
                        chunk_type=chunk_type,
                        content=current_text.strip(),
                        chapter=chapter,
                        section=section,
                        subsection=subsection,
                        page_start=sub_page or page_num,
                        page_end=sub_page or page_num,
                        source=source,
                    ))
                    chunk_idx += 1
                    sub_idx += 1
                    current_text = para
                else:
                    current_text = current_text + "\n\n" + para if current_text else para
            
            # Flush remaining
            if current_text.strip():
                sub_page = find_page_for_text(current_text, page_map) if page_map else None
                chunks.append(Chunk(
                    chunk_id=generate_chunk_id(current_text, doc_id, chunk_idx),
                    parent_id=parent_id if sub_idx > 0 else None,
                    document_id=doc_id,
                    document_type="book",
                    chunk_type=_detect_chunk_type(current_text),
                    content=current_text.strip(),
                    chapter=chapter,
                    section=section,
                    subsection=subsection,
                    page_start=sub_page or page_num,
                    page_end=sub_page or page_num,
                    source=source,
                ))
                chunk_idx += 1
    
    # Log page coverage stats
    with_pages = sum(1 for c in chunks if c.page_start is not None)
    print(f"   📄 Page numbers assigned: {with_pages}/{len(chunks)} chunks")
    
    return chunks


print("✅ Book chunker defined (with page number support)")

✅ Book chunker defined (with page number support)


In [19]:
# ═══════════════════════════════════════════════════════════
# 6B. Q&A CHUNKER — Question-aware hierarchical chunking
#     Used for both Answer Key and Student Paper
# ═══════════════════════════════════════════════════════════

def _detect_questions_from_md(md_text: str) -> list[dict]:
    """
    Detect question boundaries using multiple signals:
    1. Heading-based: ## N Question N: ...
    2. Numbered patterns: Q1., 1., 1), (1), Q.1
    3. Keyword markers: Question, Que.
    """
    questions = []
    
    # --- Strategy 1: Heading-based question detection ---
    # Matches patterns like: ## 1 Question 1: ..., ## Q1 ..., ## Question 1 ...
    heading_q_pattern = re.compile(
        r"^(#{1,3})\s+(?:(\d+)\s+)?(?:Question|Q|Que)\.?\s*(\d+)?[:\s]*(.*)",
        re.IGNORECASE | re.MULTILINE
    )
    
    # --- Strategy 2: Line-based question detection (supports optional markdown bold/italic prefix) ---
    line_q_patterns = [
        re.compile(r"^[*_]*(?:Q|Question|Que)\.?\s*(\d+)[.:\)\s*]+(.*)", re.IGNORECASE),
        re.compile(r"^[*_]*(\d+)[.\)\]]\s*(.*)"),
        re.compile(r"^[*_]*\((\d+)\)\s*(.*)"),
    ]
    
    lines = md_text.split("\n")
    
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        
        # Check heading-based
        hm = heading_q_pattern.match(stripped)
        if hm:
            q_num = hm.group(2) or hm.group(3) or str(len(questions) + 1)
            q_text = hm.group(4).strip() if hm.group(4) else ""
            questions.append({
                "question_id": f"Q{q_num}",
                "question_text": q_text,
                "line_start": i,
                "detection_method": "heading",
            })
            continue
        
        # Check line-based patterns
        for pat in line_q_patterns:
            lm = pat.match(stripped)
            if lm:
                q_num = lm.group(1)
                raw_text = lm.group(2).strip() if lm.group(2) else ""
                q_text = re.sub(r"^[*_]+\s*", "", raw_text)
                questions.append({
                    "question_id": f"Q{q_num}",
                    "question_text": q_text,
                    "line_start": i,
                    "detection_method": "line_pattern",
                })
                break
    
    # Set line_end for each question (start of next question or end of file)
    for idx, q in enumerate(questions):
        if idx + 1 < len(questions):
            q["line_end"] = questions[idx + 1]["line_start"]
        else:
            q["line_end"] = len(lines)
    
    return questions


def chunk_qa(md_text: str, doc_id: str, doc_type: str, source: str = "") -> list[Chunk]:
    """
    Question-aware hierarchical chunker for answer keys and student papers.
    
    - Each question becomes a parent chunk
    - If answer is large, it gets split into child chunks
    - All children retain question_id and parent_id
    """
    lines = md_text.split("\n")
    questions = _detect_questions_from_md(md_text)
    chunks = []
    chunk_idx = 0
    
    # Determine chunk_type based on doc_type
    answer_chunk_type = "answer" if doc_type == "answer_key" else "student_answer"
    
    for q in questions:
        q_content = "\n".join(lines[q["line_start"]:q["line_end"]]).strip()
        
        if not q_content:
            continue
        
        token_count = _estimate_tokens(q_content)
        parent_id = generate_chunk_id(q_content, doc_id, chunk_idx)
        
        if token_count <= BOOK_CHUNK_MAX_TOKENS:
            # Entire Q&A fits in one chunk
            chunks.append(Chunk(
                chunk_id=parent_id,
                parent_id=None,
                document_id=doc_id,
                document_type=doc_type,
                chunk_type="qa",
                content=q_content,
                question_id=q["question_id"],
                source=source,
            ))
            chunk_idx += 1
        else:
            # Split: question as one chunk, answer split into child chunks
            # First chunk: the question itself
            chunks.append(Chunk(
                chunk_id=parent_id,
                parent_id=None,
                document_id=doc_id,
                document_type=doc_type,
                chunk_type="question",
                content=q["question_text"] or q_content[:200],
                question_id=q["question_id"],
                source=source,
            ))
            chunk_idx += 1
            
            # Split the answer content into child chunks
            paragraphs = re.split(r"\n\n+", q_content)
            current_text = ""
            
            for para in paragraphs[1:]:  # Skip the question line
                if _estimate_tokens(current_text + "\n\n" + para) > BOOK_CHUNK_MAX_TOKENS and current_text:
                    chunks.append(Chunk(
                        chunk_id=generate_chunk_id(current_text, doc_id, chunk_idx),
                        parent_id=parent_id,
                        document_id=doc_id,
                        document_type=doc_type,
                        chunk_type=answer_chunk_type,
                        content=current_text.strip(),
                        question_id=q["question_id"],
                        source=source,
                    ))
                    chunk_idx += 1
                    current_text = para
                else:
                    current_text = current_text + "\n\n" + para if current_text else para
            
            if current_text.strip():
                chunks.append(Chunk(
                    chunk_id=generate_chunk_id(current_text, doc_id, chunk_idx),
                    parent_id=parent_id,
                    document_id=doc_id,
                    document_type=doc_type,
                    chunk_type=answer_chunk_type,
                    content=current_text.strip(),
                    question_id=q["question_id"],
                    source=source,
                ))
                chunk_idx += 1
    
    return chunks


print("✅ Q&A chunker defined")

✅ Q&A chunker defined


## 7. Load & Process MinerU Markdown Files

Load the MinerU-extracted markdown, classify, and chunk.

In [20]:
# ═══════════════════════════════════════════════════════════
# Master processing function
# NOW ACCEPTS json_path for page number extraction
# ═══════════════════════════════════════════════════════════

def process_markdown_file(md_path: str, doc_type_override: str = None, json_path: str = None) -> list[Chunk]:
    """
    Full pipeline for a single markdown file:
    1. Read markdown
    2. Classify document type (or use override)
    3. If json_path provided, build page-map for page number extraction
    4. Parse structure
    5. Apply appropriate chunking strategy (with page numbers for books)
    6. Return list of Chunks with metadata
    """
    path = Path(md_path)
    md_text = path.read_text(encoding="utf-8")
    doc_id = path.stem
    source = str(path.name)
    
    # Step 1: Classify
    doc_type = doc_type_override or classify_document(md_text, path.name)
    print(f"📄 File: {path.name}")
    print(f"   Document type: {doc_type}")
    
    # Step 2: Build page-map from JSON if available
    page_map = None
    if json_path and Path(json_path).exists():
        print(f"   📖 Loading page-map from {Path(json_path).name}...")
        page_map = build_page_map(json_path)
        print(f"   📖 Page-map built: {len(page_map)} text entries mapped to pages")
    
    # Step 3: Apply appropriate chunking
    if doc_type == "book":
        blocks = parse_markdown(md_text)
        chunks = chunk_book(blocks, doc_id, source, page_map=page_map)
    else:
        chunks = chunk_qa(md_text, doc_id, doc_type, source)
    
    print(f"   Chunks created: {len(chunks)}")
    
    # Summary
    type_counts = {}
    for c in chunks:
        type_counts[c.chunk_type] = type_counts.get(c.chunk_type, 0) + 1
    print(f"   Chunk types: {type_counts}")
    
    return chunks

In [21]:
# ═══════════════════════════════════════════════════════════
# Process the available MinerU output files
# NOW: passing JSON files for page number extraction
# ═══════════════════════════════════════════════════════════

MINERU_OUTPUT_DIR = Path("./mineru_output")

# Process the LLM Q&A file (answer_key — also has JSON for page numbers)
llm_qa_chunks = process_markdown_file(
    str(MINERU_OUTPUT_DIR / "maths_ch_8_QA_sheet.md"),
    doc_type_override="answer_key",
    json_path=str(MINERU_OUTPUT_DIR / "maths_ch_8_QA_sheet.json"),
)

print("\n" + "="*60)

# Process the book content — WITH JSON for page numbers
result_chunks = process_markdown_file(
    str(MINERU_OUTPUT_DIR / "maths_ch_8_book.md"),
    doc_type_override="book",
    json_path=str(MINERU_OUTPUT_DIR / "maths_ch_8_book.json"),
)

# Combine all chunks
all_chunks = llm_qa_chunks + result_chunks
print(f"\n📊 Total chunks across all documents: {len(all_chunks)}")

📄 File: maths_ch_8_QA_sheet.md
   Document type: answer_key
   📖 Loading page-map from maths_ch_8_QA_sheet.json...
   📖 Page-map built: 62 text entries mapped to pages
   Chunks created: 3
   Chunk types: {'question': 1, 'answer': 2}

📄 File: maths_ch_8_book.md
   Document type: book
   📖 Loading page-map from maths_ch_8_book.json...
   📖 Page-map built: 183 text entries mapped to pages
   📄 Page numbers assigned: 16/17 chunks
   Chunks created: 17
   Chunk types: {'figure': 1, 'formula': 16}

📊 Total chunks across all documents: 20


In [22]:
# ═══════════════════════════════════════════════════════════
# Preview some chunks
# ═══════════════════════════════════════════════════════════

print("\n🔍 Sample Chunks Preview:\n")
for i, chunk in enumerate(all_chunks[:5]):
    print(f"--- Chunk {i+1} ---")
    print(f"  ID:            {chunk.chunk_id}")
    print(f"  Document Type: {chunk.document_type}")
    print(f"  Chunk Type:    {chunk.chunk_type}")
    print(f"  Question ID:   {chunk.question_id}")
    print(f"  Parent ID:     {chunk.parent_id}")
    print(f"  Content:       {chunk.content[:120]}...")
    print(f"  Tokens (est):  {_estimate_tokens(chunk.content)}")
    print()


🔍 Sample Chunks Preview:

--- Chunk 1 ---
  ID:            bd2b9805cbf4cd4b04fcfb72a10c8903
  Document Type: answer_key
  Chunk Type:    question
  Question ID:   Q9
  Parent ID:     None
  Content:       Find all six trigonometric ratios for $\mathbf { 3 0 ^ { \circ } }$ and $\mathbf { 6 0 ^ { \circ } }$...
  Tokens (est):  32

--- Chunk 2 ---
  ID:            0ecdf46df937271a19d16c2b2e46c418
  Document Type: answer_key
  Chunk Type:    answer
  Question ID:   Q9
  Parent ID:     bd2b9805cbf4cd4b04fcfb72a10c8903
  Content:       ## Answer:

From the equilateral-triangle construction in the chapter:

```txt
For 30°:
sin 30° = 1/2
cos 30° = √3/2
tan...
  Tokens (est):  735

--- Chunk 3 ---
  ID:            bf03bb65a512f9ba20b2e5cf38c7f8c3
  Document Type: answer_key
  Chunk Type:    answer
  Question ID:   Q9
  Parent ID:     bd2b9805cbf4cd4b04fcfb72a10c8903
  Content:       However, the supplied chapter's worked example for tan A = 1/3 shows sec²A = 4/3, sec $A = 2 / \sqrt { 3 }$ , co

## 8. Embedding Generation

In [23]:
# Load embedding model
print("⏳ Loading embedding model...")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Embedding model loaded: {EMBEDDING_MODEL_NAME} (dim={EMBEDDING_DIM})")


def generate_embeddings(chunks: list[Chunk], batch_size: int = 32) -> list[list[float]]:
    """Generate embeddings for a list of chunks."""
    texts = [c.content for c in chunks]
    embeddings = embed_model.encode(texts, batch_size=batch_size, show_progress_bar=True)
    return embeddings.tolist()


# Generate embeddings for all chunks
print("\n⏳ Generating embeddings for all chunks...")
all_embeddings = generate_embeddings(all_chunks)
print(f"✅ Generated {len(all_embeddings)} embeddings (dim={len(all_embeddings[0])})")

⏳ Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4693.14it/s]


✅ Embedding model loaded: all-MiniLM-L6-v2 (dim=384)

⏳ Generating embeddings for all chunks...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]

✅ Generated 20 embeddings (dim=384)


## 9. Pinecone — Index Setup & Upsert

In [24]:
# ═══════════════════════════════════════════════════════════
# Initialize Pinecone
# ═══════════════════════════════════════════════════════════

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if it doesn't exist
existing_indexes = [idx.name for idx in pc.list_indexes()]

if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"⏳ Creating Pinecone index '{PINECONE_INDEX_NAME}'...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        )
    )
    print(f"✅ Index '{PINECONE_INDEX_NAME}' created")
else:
    print(f"✅ Index '{PINECONE_INDEX_NAME}' already exists")

# Connect to the index
index = pc.Index(PINECONE_INDEX_NAME)
print(f"\n📊 Index stats: {index.describe_index_stats()}")

✅ Index 'gradeforge-rag' already exists

📊 Index stats: DescribeIndexStatsResponse(dimension=384, total_vector_count=342, metric='cosine', namespaces=2)


In [25]:
# ═══════════════════════════════════════════════════════════
# Upsert chunks to Pinecone with namespace separation
# ═══════════════════════════════════════════════════════════

def get_namespace(doc_type: str) -> str:
    """Map document type to Pinecone namespace."""
    return {
        "book": NS_BOOK,
        "answer_key": NS_ANSWER_KEY,
        "student_paper": NS_STUDENT,
    }.get(doc_type, NS_BOOK)


def upsert_chunks(chunks: list[Chunk], embeddings: list[list[float]], batch_size: int = 50):
    """Upsert chunks to Pinecone, organized by namespace."""
    # Group by namespace
    ns_groups = {}
    for chunk, embedding in zip(chunks, embeddings):
        ns = get_namespace(chunk.document_type)
        if ns not in ns_groups:
            ns_groups[ns] = []
        
        # Pinecone vector record
        metadata = chunk.to_metadata()
        # Store content in metadata for retrieval (truncate if too long)
        metadata["content"] = chunk.content[:8000]  # Pinecone metadata limit
        
        ns_groups[ns].append({
            "id": chunk.chunk_id,
            "values": embedding,
            "metadata": metadata,
        })
    
    # Upsert per namespace
    for ns, vectors in ns_groups.items():
        print(f"\n⏳ Upserting {len(vectors)} vectors to namespace '{ns}'...")
        for i in range(0, len(vectors), batch_size):
            batch = vectors[i:i+batch_size]
            index.upsert(vectors=batch, namespace=ns)
        print(f"✅ Upserted {len(vectors)} vectors to '{ns}'")


# Upsert all chunks
upsert_chunks(all_chunks, all_embeddings)

print("\n📊 Updated index stats:")
print(index.describe_index_stats())


⏳ Upserting 3 vectors to namespace 'answer_key_chunks'...
✅ Upserted 3 vectors to 'answer_key_chunks'

⏳ Upserting 17 vectors to namespace 'book_chunks'...
✅ Upserted 17 vectors to 'book_chunks'

📊 Updated index stats:
DescribeIndexStatsResponse(dimension=384, total_vector_count=342, metric='cosine', namespaces=2)


## 10. Retrieval

Query Pinecone to retrieve relevant chunks for a student question.

In [26]:
def retrieve(query: str, namespaces: list[str] = None, top_k: int = TOP_K, filter_dict: dict = None, score_threshold: float = None) -> list[dict]:
    """
    Retrieve relevant chunks from Pinecone.
    
    Args:
        query: The search query text
        namespaces: Which namespaces to search (default: all)
        top_k: Number of results per namespace
        filter_dict: Optional Pinecone metadata filter
    
    Returns:
        List of result dicts with score, metadata, and content
    """
    if namespaces is None:
        namespaces = [NS_BOOK, NS_ANSWER_KEY, NS_STUDENT]
    
    # Embed the query
    query_embedding = embed_model.encode(query).tolist()
    
    all_results = []
    
    for ns in namespaces:
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            namespace=ns,
            include_metadata=True,
            filter=filter_dict,
        )
        
        for match in results.get("matches", []):
            if score_threshold is not None and match["score"] < score_threshold:
                continue
            all_results.append({
                "score": match["score"],
                "namespace": ns,
                "chunk_id": match["id"],
                "content": match["metadata"].get("content", ""),
                "metadata": match["metadata"],
            })
    
    # Sort by score descending
    all_results.sort(key=lambda x: x["score"], reverse=True)
    return all_results


# --- Test retrieval ---
test_query = "What is tokenization and why is it important for LLMs?"
results = retrieve(test_query, namespaces=[NS_ANSWER_KEY])

print(f"🔍 Query: '{test_query}'\n")
for i, r in enumerate(results[:3]):
    print(f"  Result {i+1} (score={r['score']:.4f}, ns={r['namespace']}):")
    print(f"    Question ID: {r['metadata'].get('question_id', 'N/A')}")
    print(f"    Content: {r['content'][:150]}...")
    print()

🔍 Query: 'What is tokenization and why is it important for LLMs?'

  Result 1 (score=0.8309, ns=answer_key_chunks):
    Question ID: Q1
    Content: ## 1 Question 1: What does tokenization entail, and why is it critical for LLMs?

Tokenization involves breaking down text into smaller units, or toke...

  Result 2 (score=0.5124, ns=answer_key_chunks):
    Question ID: Q10
    Content: ## 10 Question 10: What are embeddings, and how are they initialized in LLMs?

Embeddings are dense vectors that represent tokens in a continuous spac...

  Result 3 (score=0.5055, ns=answer_key_chunks):
    Question ID: Q16
    Content: ## 16 Question 16: How do LLMs manage out-of-vocabulary (OOV) words?

LLMs use subword tokenization, like Byte-Pair Encoding (BPE), to break OOV words...



## 11. Reranking

After dense retrieval, a cross-encoder reranks the candidate chunks by jointly scoring (query, chunk) pairs. This usually improves precision over bi-encoder similarity alone. The model is configured via the `RERANK_MODEL` environment variable.

In [27]:
# ═══════════════════════════════════════════════════════════
# 11. RERANKER — Cross-encoder reranking of retrieved candidates
#     Model configured via RERANK_MODEL env var
# ═══════════════════════════════════════════════════════════

from sentence_transformers import CrossEncoder

# Retrieval pool size before reranking (fetch more, then narrow down)
RERANK_CANDIDATES = 20
RERANK_TOP_K = TOP_K

print(f"⏳ Loading reranker: {RERANK_MODEL}...")
rerank_model = CrossEncoder(RERANK_MODEL)
print(f"✅ Reranker loaded: {RERANK_MODEL}")


def rerank(query: str, results: list[dict], top_k: int = RERANK_TOP_K) -> list[dict]:
    """
    Rerank a list of retrieved results using the cross-encoder.
    
    Args:
        query: The original search query
        results: List of retrieval result dicts (from retrieve())
        top_k: Number of top results to return after reranking
    
    Returns:
        Reranked list of result dicts, each with an added 'rerank_score'.
    """
    if not results:
        return []
    
    pairs = [(query, r["content"]) for r in results]
    scores = rerank_model.predict(pairs)
    
    for r, s in zip(results, scores):
        r["rerank_score"] = float(s)
    
    reranked = sorted(results, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]


def retrieve_and_rerank(
    query: str,
    namespaces: list[str] = None,
    top_k: int = RERANK_TOP_K,
    candidates: int = RERANK_CANDIDATES,
    filter_dict: dict = None,
) -> list[dict]:
    """Retrieve a wider candidate pool, then rerank down to top_k."""
    candidates_results = retrieve(
        query,
        namespaces=namespaces,
        top_k=candidates,
        filter_dict=filter_dict,
    )
    return rerank(query, candidates_results, top_k=top_k)


# --- Test reranking vs raw retrieval ---
rerank_query = "What is tokenization and why is it important for LLMs?"
raw_results = retrieve(rerank_query, namespaces=[NS_ANSWER_KEY], top_k=3)
reranked_results = retrieve_and_rerank(rerank_query, namespaces=[NS_ANSWER_KEY], top_k=3)

print(f"🔍 Query: '{rerank_query}'\n")
print("--- Raw retrieval (bi-encoder) ---")
for i, r in enumerate(raw_results):
    print(f"  {i+1}. score={r['score']:.4f} | Q={r['metadata'].get('question_id', 'N/A')}")

print("\n--- Reranked (cross-encoder) ---")
for i, r in enumerate(reranked_results):
    print(f"  {i+1}. rerank={r['rerank_score']:.4f} | dense={r['score']:.4f} | Q={r['metadata'].get('question_id', 'N/A')}")

⏳ Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3223.98it/s]


✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
🔍 Query: 'What is tokenization and why is it important for LLMs?'

--- Raw retrieval (bi-encoder) ---
  1. score=0.8309 | Q=Q1
  2. score=0.5124 | Q=Q10
  3. score=0.5055 | Q=Q16

--- Reranked (cross-encoder) ---
  1. rerank=9.2548 | dense=0.8309 | Q=Q1
  2. rerank=4.7866 | dense=0.4972 | Q=Q3
  3. rerank=4.7302 | dense=0.5055 | Q=Q16


## 12. Answer Review Agent (Groq LLM)

The evaluation agent receives grounded evidence (reference answer + book content) and evaluates the student's answer.

In [28]:
# # Initialize Groq client
# groq_client = Groq(api_key=GROQ_API_KEY)


# REVIEW_SYSTEM_PROMPT = """You are GradeForge, an expert exam answer review agent for students.

# Your task is to evaluate a student's answer by comparing it against:
# 1. The REFERENCE/CORRECT ANSWER (from the answer key)
# 2. RELEVANT STUDY MATERIAL (from the textbook, with page numbers)

# You must provide:
# - **Score** (out of 10)
# - **Correctness**: Is the answer factually correct?
# - **Completeness**: Are all key points covered?
# - **Missing Points**: What important concepts are missing?
# - **Mistakes**: Any factual errors or misconceptions?
# - **Suggestions**: How can the student improve their answer?
# - **Citations**: Reference the specific book pages where the correct information can be found.

# IMPORTANT RULES:
# - Base your evaluation ONLY on the provided reference material, NOT your internal knowledge.
# - Be fair but thorough.
# - If the student's answer is completely incorrect with no relation to the correct answer, give a score of 0.
# - If the student's answer is partially correct, give partial credit.
# - Provide actionable, specific feedback.
# - ALWAYS cite page numbers when referencing book content. Use format: (Book, p. XX) or (Book, pp. XX-YY).
# - In the Suggestions section, tell the student which pages to study for each missing point.
# """


# def review_answer(
#     question: str,
#     student_answer: str,
#     reference_answer: str = "",
#     book_context: str = "",
#     citations: str = "",
# ) -> str:
#     """
#     Use Groq LLM to review a student's answer with grounded evidence.
    
#     Follows the evaluation context from the architecture doc:
#     QUESTION + STUDENT ANSWER + REFERENCE ANSWER + RELEVANT BOOK CONTENT + CITATIONS
#     """
#     user_prompt = f"""## QUESTION
# {question}

# ## STUDENT'S ANSWER
# {student_answer}

# ## REFERENCE / CORRECT ANSWER
# {reference_answer if reference_answer else 'Not available — use study material for reference.'}

# ## RELEVANT STUDY MATERIAL (with page numbers)
# {book_context if book_context else 'Not available.'}

# ## SOURCE CITATIONS
# {citations if citations else 'No page information available.'}

# ---
# Please evaluate the student's answer and provide your detailed review.
# Include page number citations (Book, p. XX) when referencing study material.
# """
    
#     response = groq_client.chat.completions.create(
#         model=GROQ_MODEL,
#         messages=[
#             {"role": "system", "content": REVIEW_SYSTEM_PROMPT},
#             {"role": "user", "content": user_prompt},
#         ],
#         temperature=0.3,
#         max_tokens=2000,
#     )
    
#     return response.choices[0].message.content


# print("✅ Answer Review Agent ready (with citation support)")



# ============================================================
# 11. Answer Review Agent
# Primary: Gemini
# Fallback 1: Groq
# Fallback 2: Ollama
# ============================================================

# ------------------------------------------------------------
# Gemini client — PRIMARY
# ------------------------------------------------------------
from google import genai

gemini_client = None

if GEMINI_API_KEY:
    gemini_client = genai.Client(
        api_key=GEMINI_API_KEY
    )
    print(f"✅ Gemini initialized: {GEMINI_LLM_MODEL}")
else:
    print("⚠️ Gemini API key not available — Gemini disabled")


# ------------------------------------------------------------
# Groq client — FALLBACK 1
# ------------------------------------------------------------
groq_client = None

if GROQ_API_KEY:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print(f"✅ Groq fallback initialized: {GROQ_MODEL}")
else:
    print("⚠️ Groq API key not available — Groq fallback disabled")


# ------------------------------------------------------------
# System Prompt
# ------------------------------------------------------------

REVIEW_SYSTEM_PROMPT = """
You are GradeForge, an expert exam answer review agent for students.

Your task is to evaluate a student's answer by comparing it against:

1. The REFERENCE/CORRECT ANSWER from the answer key.
2. RELEVANT STUDY MATERIAL from the textbook.

You must provide:

- Score (out of 10)
- Correctness
- Completeness
- Missing Points
- Mistakes
- Suggestions
- Citations

IMPORTANT RULES & GUARDRAILS:

1. Base your evaluation ONLY on the supplied reference material. Do NOT use outside knowledge to fabricate expectations.
2. Be fair but thorough. Give partial credit when appropriate.
3. Provide actionable feedback.
4. CITATIONS RULE:
   - When referencing book content, cite book page numbers: (Book, p. XX) or (Book, pp. XX-YY).
   - In Suggestions, tell the student which pages to study for missing concepts.
5. KNOWLEDGE BASE & OUT-OF-DOMAIN GUARDRAIL:
   - If BOTH the REFERENCE ANSWER and the STUDY MATERIAL indicate that no matching reference was found (meaning the question is completely absent from the syllabus/curriculum):
     * Under Correctness: Clearly state "This question is outside the loaded knowledge base / curriculum."
     * Do NOT fabricate technical requirements from unrelated textbook chapters (e.g., do not require LLM, calculus, or unrelated concepts for a general knowledge or different subject question).
     * Under Missing Points: State "N/A - Question not found in syllabus/knowledge base."
     * Under Suggestions: State "No relevant study material found in current textbook/curriculum."
     * Under Citations: State "None (Question is outside knowledge base / curriculum)."
     * NEVER cite unrelated book titles or page numbers!
   - If relevant study material is found in the textbook, evaluate the student's answer against that study material even if an explicit reference answer key entry is absent.
"""
# REVIEW_SYSTEM_PROMPT = """
# You are GradeForge, an expert exam answer reviewer.

# Evaluate the student's answer using ONLY:
# 1. The supplied reference/correct answer.
# 2. The supplied relevant textbook material.

# Return:
# - Score /10
# - Correctness
# - Completeness
# - Missing points
# - Mistakes
# - Suggestions
# - Book citations

# Rules:
# - Do not use outside knowledge.
# - Give fair partial credit.
# - Base every judgment on the supplied material.
# - Cite textbook references as (Book, p. XX) or (Book, pp. XX-YY).
# - In Suggestions, mention relevant textbook pages for missing concepts.
# """

# ============================================================
# Gemini implementation — PRIMARY
# ============================================================

def review_with_gemini(
    system_prompt: str,
    user_prompt: str,
) -> str:

    if gemini_client is None:
        raise RuntimeError("Gemini client is not configured.")

    response = gemini_client.models.generate_content(
        model=GEMINI_LLM_MODEL,
        contents=user_prompt,
        config={
            "system_instruction": system_prompt,
            "temperature": 0.2,
        },
    )

    return response.text


# ============================================================
# Groq implementation — FALLBACK 1
# ============================================================

def review_with_groq(
    system_prompt: str,
    user_prompt: str,
) -> str:

    if groq_client is None:
        raise RuntimeError("Groq fallback is not configured.")

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0.3,
        max_tokens=2000,
    )

    return response.choices[0].message.content


# ============================================================
# Ollama implementation — FALLBACK 2 / LOCAL
# ============================================================

def review_with_ollama(
    system_prompt: str,
    user_prompt: str,
) -> str:

    response = ollama_chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        stream=False,
        options={
            "temperature": 0.2,
            "num_ctx": 1536,
            "num_predict": 800,
        },
    )

    return response.message.content


# ============================================================
# Unified GradeForge LLM
#
# Gemini → PRIMARY
# Groq   → FALLBACK 1
# Ollama  → FALLBACK 2
# ============================================================

def review_answer(
    question: str,
    student_answer: str,
    reference_answer: str = "",
    book_context: str = "",
    citations: str = "",
) -> str:

    # Determine availability for grounding
    ref_display = reference_answer.strip() if reference_answer and reference_answer.strip() else "No reference answer found in knowledge base."
    book_display = book_context.strip() if book_context and book_context.strip() else "No relevant study material found in knowledge base."
    
    if not citations or not citations.strip() or "no matching knowledge base" in citations.lower():
        cite_display = "None (Question is outside knowledge base / curriculum)."
    else:
        cite_display = citations.strip()

    user_prompt = f"""## QUESTION
{question}

## STUDENT'S ANSWER
{student_answer}

## REFERENCE / CORRECT ANSWER
{ref_display}

## RELEVANT STUDY MATERIAL (with page numbers)
{book_display}

## SOURCE CITATIONS
{cite_display}

---

Please evaluate the student's answer and provide your detailed review.
IMPORTANT: Follow all guardrails. If the question is outside the loaded knowledge base/curriculum, do NOT fabricate citations or suggestions from unrelated topics.
"""


    # ========================================================
    # PRIMARY → GEMINI
    # ========================================================

    if PRIMARY_LLM == "gemini":

        try:
            print(f"🔵 Calling Gemini: {GEMINI_LLM_MODEL}")

            result = review_with_gemini(
                system_prompt=REVIEW_SYSTEM_PROMPT,
                user_prompt=user_prompt,
            )

            print("✅ Gemini evaluation completed")

            return result

        except Exception as gemini_error:

            print("\n⚠️ Gemini failed!")
            print(f"   Error: {gemini_error}")


            # =================================================
            # FALLBACK 1 → GROQ
            # =================================================

            if ENABLE_GROQ_FALLBACK and groq_client is not None:

                try:
                    print(f"\n🔄 Falling back to Groq: {GROQ_MODEL}")

                    result = review_with_groq(
                        system_prompt=REVIEW_SYSTEM_PROMPT,
                        user_prompt=user_prompt,
                    )

                    print("✅ Groq fallback evaluation completed")

                    return result

                except Exception as groq_error:

                    print("\n⚠️ Groq fallback failed!")
                    print(f"   Error: {groq_error}")

            else:
                groq_error = "Groq fallback unavailable"


            # =================================================
            # FALLBACK 2 → OLLAMA
            # =================================================

            if ENABLE_OLLAMA_FALLBACK:

                try:
                    print(f"\n🦙 Falling back to Ollama: {OLLAMA_MODEL}")

                    result = review_with_ollama(
                        system_prompt=REVIEW_SYSTEM_PROMPT,
                        user_prompt=user_prompt,
                    )

                    print("✅ Ollama fallback evaluation completed")

                    return result

                except Exception as ollama_error:

                    raise RuntimeError(
                        f"""
All LLM providers failed.

Gemini error:
{gemini_error}

Groq error:
{groq_error}

Ollama error:
{ollama_error}
"""
                    )

            raise RuntimeError(
                f"Gemini failed and no fallback was available:\n{gemini_error}"
            )


    # ========================================================
    # OPTIONAL → GROQ PRIMARY
    # ========================================================

    elif PRIMARY_LLM == "groq":

        try:
            print(f"🟢 Calling Groq: {GROQ_MODEL}")

            return review_with_groq(
                system_prompt=REVIEW_SYSTEM_PROMPT,
                user_prompt=user_prompt,
            )

        except Exception as groq_error:

            print("\n⚠️ Groq failed!")
            print(f"   Error: {groq_error}")

            if ENABLE_OLLAMA_FALLBACK:

                print(
                    f"\n🦙 Falling back to Ollama: {OLLAMA_MODEL}"
                )

                return review_with_ollama(
                    system_prompt=REVIEW_SYSTEM_PROMPT,
                    user_prompt=user_prompt,
                )

            raise


    # ========================================================
    # OPTIONAL → OLLAMA PRIMARY
    # ========================================================

    elif PRIMARY_LLM == "ollama":

        try:
            print(f"🦙 Calling Ollama: {OLLAMA_MODEL}")

            return review_with_ollama(
                system_prompt=REVIEW_SYSTEM_PROMPT,
                user_prompt=user_prompt,
            )

        except Exception as ollama_error:

            print("\n⚠️ Ollama failed!")
            print(f"   Error: {ollama_error}")

            if ENABLE_GROQ_FALLBACK and groq_client is not None:

                print(
                    f"\n🔄 Falling back to Groq: {GROQ_MODEL}"
                )

                return review_with_groq(
                    system_prompt=REVIEW_SYSTEM_PROMPT,
                    user_prompt=user_prompt,
                )

            raise


    else:

        raise ValueError(
            f"Unsupported PRIMARY_LLM: {PRIMARY_LLM}"
        )


# ============================================================
# Ready
# ============================================================

print("✅ GradeForge Answer Review Agent ready")
print(f"🔵 Primary: Gemini → {GEMINI_LLM_MODEL}")
print(f"🟢 Fallback 1: Groq → {GROQ_MODEL}")
print(f"🦙 Fallback 2: Ollama → {OLLAMA_MODEL}")

✅ Gemini initialized: gemini-3-flash-preview
✅ Groq fallback initialized: openai/gpt-oss-120b
✅ GradeForge Answer Review Agent ready
🔵 Primary: Gemini → gemini-3-flash-preview
🟢 Fallback 1: Groq → openai/gpt-oss-120b
🦙 Fallback 2: Ollama → qwen3:4b


## 13. Full Pipeline Demo — Review a Student Answer

Putting it all together: question → retrieve context → evaluate with LLM

In [29]:
import os
import re
import time
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable, Image, PageBreak
)

CACHE_DIR = "formula_cache"
# Register DejaVu Sans so Unicode math symbols (√, ∑, ∂, ⊤ ...) render in PDF captions
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
_DEJAVU_DIR = os.path.join(os.path.dirname(matplotlib.__file__), 'mpl-data', 'fonts', 'ttf')
try:
    pdfmetrics.registerFont(TTFont('DejaVuSans', os.path.join(_DEJAVU_DIR, 'DejaVuSans.ttf')))
    pdfmetrics.registerFont(TTFont('DejaVuSans-Oblique', os.path.join(_DEJAVU_DIR, 'DejaVuSans-Oblique.ttf')))
    pdfmetrics.registerFont(TTFont('DejaVuSans-Bold', os.path.join(_DEJAVU_DIR, 'DejaVuSans-Bold.ttf')))
    pdfmetrics.registerFontFamily('DejaVuSans', normal='DejaVuSans',
                                  bold='DejaVuSans-Bold', italic='DejaVuSans-Oblique',
                                  boldItalic='DejaVuSans-Bold')
    _UNICODE_FONT = 'DejaVuSans'
except Exception:
    _UNICODE_FONT = 'Courier'

os.makedirs(CACHE_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════
# FORMULA RENDERING ENGINE (fixed)
#   - Renders EVERY math segment (display + inline) to PNG
#   - Always prints the readable formula string underneath (both)
#   - Robust LaTeX normalization + unicode fallback (no crash)
# ═══════════════════════════════════════════════════════════
import os
import re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image as PILImage

try:
    from pylatexenc.latex2text import LatexNodes2Text as _LatexNodes2Text
    _latex_to_unicode = _LatexNodes2Text().latex_to_text
except Exception:
    _latex_to_unicode = None


def _strip_math_delimiters(s: str) -> str:
    """Remove surrounding math delimiters ($$, \\[ \\], \\( \\), $)."""
    s = s.strip()
    for d in ('$$', '\\[', '\\]', '\\(', '\\)', '$'):
        if s.startswith(d):
            s = s[len(d):]
        if s.endswith(d):
            s = s[:-len(d)]
    return s.strip()


def _normalize_latex_for_mathtext(s: str) -> str:
    """Normalize MinerU/LLM LaTeX into something matplotlib mathtext can parse."""
    s = s.strip()
    s = re.sub(r'\\tag\{[^}]*\}', '', s)
    s = re.sub(r'\\label\{[^}]*\}', '', s)
    s = s.replace('&', '')
    s = s.replace('\\operatorname*', '\\mathrm').replace('\\operatorname', '\\mathrm')
    s = s.replace('\\left\\{', '\\{').replace('\\right\\}', '\\}')
    s = s.replace('\\right.', '')
    s = s.replace('\\left|', '|').replace('\\right|', '|')
    s = s.replace('\\left.', '')
    s = s.replace('\\left(', '(').replace('\\right)', ')')
    s = s.replace('\\left[', '[').replace('\\right]', ']')
    s = s.replace('\\left', '').replace('\\right', '')
    s = s.replace('\\!', ' ').replace('\\,', ' ').replace('\\;', ' ')
    s = s.replace('\\Big', '').replace('\\big', '')
    s = s.replace('\\displaystyle', '')
    s = re.sub(r'\\begin\{array\}\{[^}]*\}', '', s)
    s = s.replace('\\end{array}', '')
    s = s.replace('\\\\', ' ; ')
    s = s.replace('\\blacksquare', '\\square')
    return s.strip()


def _latex_to_plain(s: str) -> str:
    """Convert LaTeX to a readable plain/unicode string (always available)."""
    raw = _strip_math_delimiters(s)
    if _latex_to_unicode is not None:
        try:
            return _latex_to_unicode(raw).replace('\n', ' ')
        except Exception:
            pass
    return raw.replace('\\', '')


def render_formula_to_image(latex_expr: str, prefix: str = "eq") -> str:
    """
    Render a LaTeX formula (with or without delimiters) into a crisp PNG.
    Attempt 1: matplotlib mathtext (true formula).
    Attempt 2: unicode plain-text render (still an image, still readable).
    Returns path to PNG, or None only if even the fallback fails.
    """
    clean = _strip_math_delimiters(latex_expr)
    if not clean:
        return None

    norm = _normalize_latex_for_mathtext(clean)
    filename = f"{prefix}_{abs(hash(norm)) % 1000000}.png"
    out_path = os.path.join(CACHE_DIR, filename)
    if os.path.exists(out_path):
        return out_path

    # Attempt 1: mathtext
    try:
        fig = plt.figure(figsize=(0.01, 0.01), dpi=220)
        t = fig.text(0, 0, f"${norm}$", fontsize=13)
        fig.canvas.draw()
        bb = t.get_window_extent()
        w_in, h_in = bb.width / fig.dpi, bb.height / fig.dpi
        plt.close(fig)
        fig = plt.figure(figsize=(w_in + 0.25, h_in + 0.25), dpi=220)
        fig.text(0.5, 0.5, f"${norm}$", fontsize=13, ha='center', va='center')
        plt.axis('off')
        fig.savefig(out_path, bbox_inches='tight', transparent=True, pad_inches=0.03)
        plt.close(fig)
        return out_path
    except Exception:
        try:
            plt.close('all')
        except Exception:
            pass

    # Attempt 2: unicode fallback (never blocks the report)
    plain = _latex_to_plain(clean).replace('\n', ' ').strip()
    if not plain:
        return None
    try:
        fig = plt.figure(figsize=(0.01, 0.01), dpi=220)
        t = fig.text(0, 0, plain, fontsize=13, fontname='DejaVu Sans')
        fig.canvas.draw()
        bb = t.get_window_extent()
        w_in, h_in = bb.width / fig.dpi, bb.height / fig.dpi
        plt.close(fig)
        fig = plt.figure(figsize=(w_in + 0.25, h_in + 0.25), dpi=220)
        fig.text(0.5, 0.5, plain, fontsize=13, ha='center', va='center', fontname='DejaVu Sans')
        plt.axis('off')
        fig.savefig(out_path, bbox_inches='tight', transparent=True, pad_inches=0.03)
        plt.close(fig)
        return out_path
    except Exception:
        try:
            plt.close('all')
        except Exception:
            pass
        return None


def _formula_image_flowable(img_path: str, max_width: float = 470.0, max_height: float = 110.0):
    """Embed a formula PNG preserving its true aspect ratio (no squashing)."""
    try:
        with PILImage.open(img_path) as im:
            iw, ih = im.size
    except Exception:
        iw, ih = 4, 1
    aspect = (ih / iw) if iw else 0.25
    w = max_width
    h = w * aspect
    if h > max_height:
        h = max_height
        w = h / aspect if aspect else max_width
    if h < 14:
        h = 14
    img = Image(img_path, width=w, height=h)
    img.hAlign = 'CENTER'
    return img


_formula_style_cache = {}


def _formula_caption_style():
    if 'style' not in _formula_style_cache:
        _formula_style_cache['style'] = ParagraphStyle(
            'FormulaCaption',
            fontName=_UNICODE_FONT,
            fontSize=8,
            leading=10,
            textColor=colors.HexColor('#6B7280'),
            spaceBefore=1,
            spaceAfter=1,
        )
    return _formula_style_cache['style']


def build_formula_block(latex_expr: str) -> list:
    """Return flowables for a DISPLAY formula: rendered image + readable string (both)."""
    block = []
    plain = _latex_to_plain(latex_expr).replace('\n', ' ').strip()
    img_path = render_formula_to_image(latex_expr)
    if img_path and os.path.exists(img_path):
        block.append(Spacer(1, 2))
        block.append(_formula_image_flowable(img_path))
    if plain:
        block.append(Paragraph(
            f"Formula: <font name={_UNICODE_FONT} size='7.5'>{clean_text_for_xml(plain)}</font>",
            _formula_caption_style(),
        ))
    elif latex_expr.strip():
        block.append(Paragraph(
            f"<font name={_UNICODE_FONT} size='7.5'>{clean_text_for_xml(latex_expr.strip())}</font>",
            _formula_caption_style(),
        ))
    block.append(Spacer(1, 4))
    return block


_MATH_TOKEN_RE = re.compile(
    r'(\$\$.+?\$\$|\\\[.+?\\\]|\\\(.+?\\\)|(?<!\$)\$(?!\$)[^$\n]+?\$(?!\$))',
    re.DOTALL,
)


def _inline_image_tag(img_path: str, disp_h: float = 11.0) -> str:
    """Build a ReportLab inline <img> tag sized to a text-line height."""
    try:
        with PILImage.open(img_path) as im:
            iw, ih = im.size
        disp_w = disp_h * iw / ih if ih else disp_h
        return f'<img src="{img_path}" width="{disp_w:.1f}" height="{disp_h:.1f}" valign="-2"/>'
    except Exception:
        return ''


def expand_math_inline(text: str, base_style) -> list:
    """
    Split a line into plain-text paragraphs and rendered math.
    Inline math -> embedded <img> in the running sentence.
    Display math -> centered formula image + readable string underneath.
    """
    if not text or not re.search(r'\$|\\\(|\\\[', text):
        return [Paragraph(clean_text_for_xml(text), base_style)] if text.strip() else []

    parts = _MATH_TOKEN_RE.split(text)
    flowables = []
    buffer = ""
    for part in parts:
        if not part:
            continue
        if _MATH_TOKEN_RE.fullmatch(part):
            if buffer.strip():
                flowables.append(Paragraph(buffer.strip(), base_style))
                buffer = ""
            token = part.strip()
            display = token.startswith('$$') or token.startswith('\\[')
            img_path = render_formula_to_image(part, prefix="inl")
            cap = _latex_to_plain(part).replace('\n', ' ').strip()
            if img_path and os.path.exists(img_path):
                if display:
                    flowables.append(Spacer(1, 2))
                    flowables.append(_formula_image_flowable(img_path, max_width=440, max_height=90))
                    flowables.append(Paragraph(
                        f"Formula: <font name={_UNICODE_FONT} size='7.5'>{clean_text_for_xml(cap)}</font>",
                        _formula_caption_style(),
                    ))
                    flowables.append(Spacer(1, 3))
                else:
                    buffer += _inline_image_tag(img_path, disp_h=11.0)
            else:
                buffer += f" Formula: {clean_text_for_xml(cap)} "
        else:
            buffer += clean_text_for_xml(part)
    if buffer.strip():
        flowables.append(Paragraph(buffer.strip(), base_style))
    return flowables


def _inline_formula_markup(text: str) -> str:
    """Single-string variant (for Table cells / question-answer boxes)."""
    if not text or not re.search(r'\$|\\\(|\\\[', text):
        return clean_text_for_xml(text)
    parts = _MATH_TOKEN_RE.split(text)
    out = ""
    for part in parts:
        if not part:
            continue
        if _MATH_TOKEN_RE.fullmatch(part):
            img_path = render_formula_to_image(part, prefix="inl")
            cap = _latex_to_plain(part).replace('\n', ' ').strip()
            if img_path and os.path.exists(img_path):
                out += _inline_image_tag(img_path, disp_h=11.0)
            else:
                out += f" Formula: {clean_text_for_xml(cap)} "
        else:
            out += clean_text_for_xml(part)
    return out

def clean_text_for_xml(text: str) -> str:
    """Escape special characters for ReportLab XML parser and normalize Unicode."""
    text = text.replace("&", "&amp;")
    text = text.replace("<", "&lt;")
    text = text.replace(">", "&gt;")
    # Normalize ring operator and weird degree symbols to standard degree symbol
    text = text.replace('\u2218', '°')
    text = text.replace('\u200b', '') # strip zero-width space
    text = text.replace('\ufeff', '')
    text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)
    text = re.sub(r'\*(.*?)\*', r'<i>\1</i>', text)
    text = re.sub(r'^#+\s*', '', text)
    return text


def parse_markdown_to_flowables(md_text: str, styles) -> list:
    """Convert markdown review text into ReportLab Flowables with rendered formula images."""
    flowables = []
    lines = md_text.strip().split('\n')

    heading_style = ParagraphStyle(
        'ReviewHeading',
        parent=styles['Heading2'],
        fontSize=11.5,
        leading=15,
        textColor=colors.HexColor('#1E3A8A'),
        spaceBefore=8,
        spaceAfter=3,
        keepWithNext=True
    )

    subheading_style = ParagraphStyle(
        'ReviewSubheading',
        parent=styles['Heading3'],
        fontSize=10,
        leading=13.5,
        textColor=colors.HexColor('#1E40AF'),
        spaceBefore=7,
        spaceAfter=3,
        keepWithNext=True
    )

    body_style = ParagraphStyle(
        'ReviewBody',
        parent=styles['Normal'],
        fontName=_UNICODE_FONT,
        fontSize=9,
        leading=13,
        textColor=colors.HexColor('#1F2937'),
        spaceAfter=3
    )

    bullet_style = ParagraphStyle(
        'ReviewBullet',
        parent=body_style,
        leftIndent=15,
        firstLineIndent=-10,
        spaceAfter=2.5
    )

    i = 0
    while i < len(lines):
        line_str = lines[i].strip()
        if not line_str or line_str == '---':
            flowables.append(Spacer(1, 3))
            i += 1
            continue

        # --- Multi-line display math: opening $$ or \[ on its own line ---
        if line_str in ('$$', '\\['):
            formula_lines = []
            i += 1
            while i < len(lines) and lines[i].strip() not in ('$$', '\\]'):
                formula_lines.append(lines[i].strip())
                i += 1
            i += 1  # consume closing delimiter line
            formula_code = ' '.join(formula_lines).strip()
            if formula_code:
                flowables.extend(build_formula_block(formula_code))
            continue

        # --- Headings / bullets (structure) ---
        if line_str.startswith('### '):
            flowables.append(Paragraph(clean_text_for_xml(line_str[4:].strip()), subheading_style))
        elif line_str.startswith('## ') or line_str.startswith('# '):
            flowables.append(Paragraph(clean_text_for_xml(line_str.lstrip('#').strip()), heading_style))
        elif line_str.startswith('- ') or line_str.startswith('* ') or line_str.startswith('• '):
            flowables.extend(expand_math_inline('- ' + line_str[2:].strip(), bullet_style))
        elif re.match(r'^\d+\.\s+', line_str):
            flowables.extend(expand_math_inline(line_str, bullet_style))
        else:
            # Any remaining line: render math inline, render display math as blocks.
            flowables.extend(expand_math_inline(line_str, body_style))

        i += 1

    return flowables

def _format_citations(results: list[dict]) -> str:
    """Format retrieved chunk metadata into citation strings."""
    if not results:
        return "No matching knowledge base reference found."
    citations = []
    for i, r in enumerate(results):
        meta = r.get('metadata', {})
        parts = [f"[Source {i+1}]"]
        
        source = meta.get('source', '')
        if source:
            parts.append(f"Book: {source}")
        
        chapter = meta.get('chapter', '')
        if chapter:
            parts.append(f"Chapter: {chapter}")
        
        section = meta.get('section', '')
        if section:
            parts.append(f"Section: {section}")
        
        page = meta.get('page_start')
        page_end = meta.get('page_end')
        if page:
            if page_end and page_end != page:
                parts.append(f"Pages: {page}-{page_end}")
            else:
                parts.append(f"Page: {page}")
        
        citations.append(' | '.join(parts))
    
    return '\n'.join(citations) if citations else "No matching knowledge base reference found."


def _format_book_context_with_pages(results: list[dict]) -> str:
    """Format book context with inline page references."""
    if not results:
        return "No relevant study material found in knowledge base."
    sections = []
    for r in results:
        meta = r.get('metadata', {})
        page = meta.get('page_start')
        section = meta.get('section', '')
        chapter = meta.get('chapter', '')
        
        header_parts = []
        if chapter:
            header_parts.append(chapter)
        if section:
            header_parts.append(section)
        if page:
            header_parts.append(f"p. {page}")
        
        header = ' > '.join(header_parts) if header_parts else 'Unknown source'
        sections.append(f"### [{header}]\n{r['content']}")
    
    return '\n\n'.join(sections)


def generate_review_pdf(
    question: str,
    student_answer: str,
    review_text: str,
    reference_answer: str = "",
    citations: str = "",
    output_filename: str = "student_review.pdf"
) -> str:
    """Generate a single question PDF assessment report with formula image rendering."""
    doc = SimpleDocTemplate(
        output_filename,
        pagesize=letter,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )
    
    styles = getSampleStyleSheet()
    story = []
    
    score_match = re.search(r'(?:Score|score)[:\s*]+([0-9]+(?:\.[0-9]+)?)\s*[/]\s*10', review_text, re.IGNORECASE)
    score_val = score_match.group(1) if score_match else "N/A"
    
    score_color = colors.HexColor('#10B981')
    if score_val != "N/A":
        try:
            s = float(score_val)
            if s <= 4:
                score_color = colors.HexColor('#EF4444')
            elif s < 8:
                score_color = colors.HexColor('#F59E0B')
        except ValueError:
            pass

    title_style = ParagraphStyle(
        'HeaderTitle',
        parent=styles['Heading1'],
        fontSize=18,
        leading=22,
        textColor=colors.HexColor('#1E3A8A'),
        fontName='Helvetica-Bold'
    )
    subtitle_style = ParagraphStyle(
        'HeaderSubtitle',
        parent=styles['Normal'],
        fontSize=9,
        leading=12,
        textColor=colors.HexColor('#6B7280')
    )
    score_box_style = ParagraphStyle(
        'ScoreBox',
        parent=styles['Normal'],
        fontSize=15,
        leading=17,
        alignment=1,
        textColor=colors.white,
        fontName='Helvetica-Bold'
    )
    score_sub_style = ParagraphStyle(
        'ScoreSub',
        parent=styles['Normal'],
        fontSize=8,
        leading=10,
        alignment=1,
        textColor=colors.white
    )

    header_left = [
        Paragraph("GradeForge Assessment Report", title_style),
        Spacer(1, 2),
        Paragraph(f"Generated: {datetime.now().strftime('%b %d, %Y - %H:%M')} | Automated AI Review", subtitle_style)
    ]
    
    header_right = [
        Paragraph(f"SCORE: {score_val} / 10", score_box_style),
        Paragraph("Evaluation Result", score_sub_style)
    ]
    
    header_table = Table([[header_left, header_right]], colWidths=[400, 130])
    header_table.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BACKGROUND', (1,0), (1,0), score_color),
        ('ROUNDEDCORNERS', [4, 4, 4, 4]),
        ('TOPPADDING', (1,0), (1,0), 6),
        ('BOTTOMPADDING', (1,0), (1,0), 6),
    ]))
    story.append(header_table)
    story.append(Spacer(1, 10))
    story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#E5E7EB'), spaceBefore=2, spaceAfter=10))

    section_hdr_style = ParagraphStyle(
        'SectionHeader',
        parent=styles['Heading2'],
        fontSize=11,
        leading=14,
        textColor=colors.HexColor('#1E40AF'),
        fontName='Helvetica-Bold',
        spaceBefore=6,
        spaceAfter=4,
        keepWithNext=True
    )
    box_body_style = ParagraphStyle(
        'BoxBody',
        parent=styles['Normal'],
        fontName=_UNICODE_FONT,
        fontSize=9.5,
        leading=13.5,
        textColor=colors.HexColor('#111827')
    )

    # Question Box
    q_cell = [
        Paragraph("<b>Question:</b>", section_hdr_style),
        Paragraph(clean_text_for_xml(question), box_body_style)
    ]
    q_table = Table([[q_cell]], colWidths=[530])
    q_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#F3F4F6')),
        ('ROUNDEDCORNERS', [4, 4, 4, 4]),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
        ('RIGHTPADDING', (0,0), (-1,-1), 10),
        ('TOPPADDING', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
    ]))
    story.append(q_table)
    story.append(Spacer(1, 8))

    # Student Answer Box
    ans_cell = [
        Paragraph("<b>Student's Answer:</b>", section_hdr_style),
        Paragraph(clean_text_for_xml(student_answer.strip()), box_body_style)
    ]
    ans_table = Table([[ans_cell]], colWidths=[530])
    ans_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#EFF6FF')),
        ('ROUNDEDCORNERS', [4, 4, 4, 4]),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
        ('RIGHTPADDING', (0,0), (-1,-1), 10),
        ('TOPPADDING', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
    ]))
    story.append(ans_table)
    story.append(Spacer(1, 10))

    # Citations Box
    if citations:
        is_no_kb = "no matching knowledge base" in citations.lower() or citations.strip() == "None"
        box_bg = colors.HexColor('#F3F4F6') if is_no_kb else colors.HexColor('#FEF3C7')
        cite_cell = [
            Paragraph("<b>Recommended Study Material & Book Citations:</b>", section_hdr_style),
            Spacer(1, 3),
            Paragraph(clean_text_for_xml(citations).replace('\n', '<br/>'), box_body_style)
        ]
        cite_table = Table([[cite_cell]], colWidths=[530])
        cite_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), box_bg),
            ('ROUNDEDCORNERS', [4, 4, 4, 4]),
            ('LEFTPADDING', (0,0), (-1,-1), 10),
            ('RIGHTPADDING', (0,0), (-1,-1), 10),
            ('TOPPADDING', (0,0), (-1,-1), 8),
            ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ]))
        story.append(cite_table)
        story.append(Spacer(1, 10))

    # Evaluation with Formula Images
    story.append(Paragraph("Detailed Evaluation & Feedback", section_hdr_style))
    story.append(Spacer(1, 4))
    
    review_flowables = parse_markdown_to_flowables(review_text, styles)
    story.extend(review_flowables)

    doc.build(story)
    return output_filename


def grade_student_answer(
    question: str, 
    student_answer: str, 
    generate_pdf: bool = False, 
    output_filename: str = "student_review.pdf",
    score_threshold: float = SIMILARITY_THRESHOLD
) -> str:
    """Full GradeForge pipeline for evaluating a single student question with guardrails."""
    print("="*60)
    print(f"📝 QUESTION: {question}")
    print(f"✍️  STUDENT ANSWER: {student_answer[:200]}..." if len(student_answer) > 200 else f"✍️  STUDENT ANSWER: {student_answer}")
    print("="*60)
    
    # Step 1: Retrieve reference answer
    print(f"\n🔍 Step 1: Retrieving reference answer (threshold={score_threshold})...")
    ref_results = retrieve(question, namespaces=[NS_ANSWER_KEY], top_k=3, score_threshold=score_threshold)
    reference_answer = "\n\n".join([r["content"] for r in ref_results[:2]]) if ref_results else ""
    print(f"   Found {len(ref_results)} reference chunks above threshold")
    
    # Step 2: Retrieve book context
    print(f"\n🔍 Step 2: Retrieving book context (threshold={score_threshold})...")
    book_results = retrieve(question, namespaces=[NS_BOOK], top_k=3, score_threshold=score_threshold)
    book_context = _format_book_context_with_pages(book_results[:3]) if book_results else ""
    print(f"   Found {len(book_results)} book chunks above threshold")
    
    if book_results:
        for r in book_results[:3]:
            meta = r.get('metadata', {})
            pg = meta.get('page_start', '?')
            sec = meta.get('section', meta.get('chapter', 'N/A'))
            print(f"   📄 Page {pg} — {sec} (score: {r['score']:.4f})")
    else:
        print("   ⚠️ No book chunks met relevance threshold — question is outside knowledge base.")
    
    # Step 3: Format citations
    citations = _format_citations(book_results[:3])
    
    # Step 4: Review with LLM
    print("\n🤖 Step 3: Evaluating with GradeForge Agent...")
    review = review_answer(
        question=question,
        student_answer=student_answer,
        reference_answer=reference_answer,
        book_context=book_context,
        citations=citations,
    )
    
    print("\n" + "="*60)
    print("📊 EVALUATION RESULT")
    print("="*60)
    print(review)

    # Step 5: (Optional) Generate PDF
    if generate_pdf:
        print(f"\n📄 Generating assessment report PDF: {output_filename}...")
        try:
            generate_review_pdf(
                question=question,
                student_answer=student_answer,
                review_text=review,
                reference_answer=reference_answer,
                citations=citations,
                output_filename=output_filename,
            )
            print(f"✅ PDF report successfully saved as: {output_filename}")
        except Exception as e:
            print(f"⚠️ Failed to generate PDF: {e}")
            import traceback
            traceback.print_exc()
    
    return review

In [ ]:
# ═══════════════════════════════════════════════════════════
# DEMO: Grade a sample student answer
# ═══════════════════════════════════════════════════════════

sample_question = "What is trigonometry? Explain its meaning and applications mentioned in the chapter."

sample_student_answer = """
Trigonometry is bournab of mathematics that studies solution between the sides and angle of

a triangle. The chapter's explains the trigenometry can be used to find heights and distances that ere difficult to measure directly, such as height of monument.
"""

review = grade_student_answer(sample_question, sample_student_answer,generate_pdf=True,
    output_filename="review_test\maths_ch_8_q1_handwritten_revirew_run_3.pdf")

<>:14: SyntaxWarning: invalid escape sequence '\m'
<>:14: SyntaxWarning: invalid escape sequence '\m'
C:\Users\IA Dev86\AppData\Local\Temp\ipykernel_24680\568825567.py:14: SyntaxWarning: invalid escape sequence '\m'
  output_filename="review_test\maths_ch_8_q1_handwritten_revirew_run_3.pdf")


📝 QUESTION: What is trigonometry? Explain its meaning and applications mentioned in the chapter.
✍️  STUDENT ANSWER: 
Trigonometry is bournab of mathematics that studies solution between the sides and angle of

a triangle. The chapter's explains the trigenometry can be used to find heights and distances that ere dif...

🔍 Step 1: Retrieving reference answer...
   Found 3 reference chunks

🔍 Step 2: Retrieving book context...
   Found 2 book chunks
   📄 Page 8 — 8.5 Summary
   📄 Page 9 — 8.3 Trigonometric Ratios of Some Specific Angles

🤖 Step 3: Evaluating with GradeForge Agent...
🔵 Calling Gemini: gemini-3-flash-preview
✅ Gemini evaluation completed

📊 EVALUATION RESULT
### **GradeForge Evaluation**

**Score: 7/10**

---

### **Correctness**
The student correctly identifies the core essence of trigonometry as the study of the relationship between the sides and angles of a triangle. The student also accurately identifies a primary application mentioned in the context of the chapter: ca

In [31]:
# ═══════════════════════════════════════════════════════════
# DEMO 2: Another example
# ═══════════════════════════════════════════════════════════

sample_question_2 = "Find the value of sin45°, cos45°, tan45° cosec 45°, sec 45° and cot 45°"

sample_student_answer_2 = """
 In a right rectangle with one acute angle 
45
∘
45 
∘
 , the order other acute angle is also 
45
∘
45 
∘
 . Hence the two legs are equal

let each leg be a. Pythagoras. hypotenuse = 
9
2
9 
2
​
 

Therefore: 
sin
⁡
45
∘
=
a
/
(
a
2
)
=
1
/
2
sin45 
∘
 =a/(a 
2
​
 )=1/ 
2
​
  
cos
⁡
45
∘
=
a
/
(
a
2
)
=
1
/
2
cos45 
∘
 =a/(a 
2
​
 )=1/ 
2
​
  
tan
⁡
45
∘
=
a
/
a
=
1
tan45 
∘
 =a/a=1 
csc
⁡
45
∘
=
2
csc45 
∘
 = 
2
​
  
sec
⁡
45
∘
=
2
sec45 
∘
 = 
2
​
  
cot
⁡
45
∘
=
2
cot45 
∘
 = 
2
​


"""

review_2 = grade_student_answer(sample_question_2, sample_student_answer_2,generate_pdf=True,
    output_filename="review_test\maths_ch_8_q1_corrected_run_2.pdf")

<>:130: SyntaxWarning: invalid escape sequence '\m'
<>:130: SyntaxWarning: invalid escape sequence '\m'
C:\Users\IA Dev86\AppData\Local\Temp\ipykernel_18624\907508304.py:130: SyntaxWarning: invalid escape sequence '\m'
  output_filename="review_test\maths_ch_8_q1_corrected_run_2.pdf")


📝 QUESTION: Find the value of sin45°, cos45°, tan45° cosec 45°, sec 45° and cot 45°
✍️  STUDENT ANSWER: 
 In a right rectangle with one acute angle 
45
∘
45 
∘
 , the order other acute angle is also 
45
∘
45 
∘
 . Hence the two legs are equal

let each leg be a. Pythagoras. hypotenuse = 
9
2
9 
2
​


Th...

🔍 Step 1: Retrieving reference answer (threshold=0.3)...
   Found 0 reference chunks above threshold

🔍 Step 2: Retrieving book context (threshold=0.3)...
   Found 3 book chunks above threshold
   📄 Page 9 — EXERCISE 8.1 (score: 0.6055)
   📄 Page 15 — EXERCISE 8.2 (score: 0.5398)
   📄 Page 19 — EXERCISE 8.3 (score: 0.4951)

🤖 Step 3: Evaluating with GradeForge Agent...
🔵 Calling Gemini: gemini-3-flash-preview
✅ Gemini evaluation completed

📊 EVALUATION RESULT
- **Score:** 0/10
- **Correctness:** This question is outside the loaded knowledge base / curriculum.
- **Completeness:** N/A
- **Missing Points:** N/A - Question not found in syllabus/knowledge base.
- **Mistakes:** N/A
- **Su

In [30]:
# ═══════════════════════════════════════════════════════════
# DEMO 2: Another example
# ═══════════════════════════════════════════════════════════

sample_question_2 = "how was mahatma gandhi"

sample_student_answer_2 = """
he was freedom figher from india.
"""

review_2 = grade_student_answer(sample_question_2, sample_student_answer_2,generate_pdf=True,
    output_filename="review_test\out_of_box_quation_review.pdf")

<>:12: SyntaxWarning: invalid escape sequence '\o'
<>:12: SyntaxWarning: invalid escape sequence '\o'
C:\Users\IA Dev86\AppData\Local\Temp\ipykernel_18624\2963276203.py:12: SyntaxWarning: invalid escape sequence '\o'
  output_filename="review_test\out_of_box_quation_review.pdf")


📝 QUESTION: how was mahatma gandhi
✍️  STUDENT ANSWER: 
he was freedom figher from india.


🔍 Step 1: Retrieving reference answer (threshold=0.3)...
   Found 0 reference chunks above threshold

🔍 Step 2: Retrieving book context (threshold=0.3)...


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


   Found 0 book chunks above threshold
   ⚠️ No book chunks met relevance threshold — question is outside knowledge base.

🤖 Step 3: Evaluating with GradeForge Agent...
🔵 Calling Gemini: gemini-3-flash-preview
✅ Gemini evaluation completed

📊 EVALUATION RESULT
**Score:** 0/10 (Cannot be evaluated)

**Correctness:** 
This question is outside the loaded knowledge base / curriculum.

**Completeness:** 
N/A

**Missing Points:** 
N/A - Question not found in syllabus/knowledge base.

**Mistakes:** 
N/A

**Suggestions:** 
No relevant study material found in current textbook/curriculum.

**Citations:** 
None (Question is outside knowledge base / curriculum).

📄 Generating assessment report PDF: review_test\out_of_box_quation_review.pdf...
✅ PDF report successfully saved as: review_test\out_of_box_quation_review.pdf


## 14. Utility — Process Any New Markdown File

Use this cell to process and index any new markdown file.

In [ ]:
def ingest_new_document(md_path: str, doc_type: str = None):
    """
    Ingest a new markdown file into the RAG pipeline.
    
    Args:
        md_path: Path to the markdown file
        doc_type: Override document type ('book', 'answer_key', 'student_paper')
    """
    # 1. Parse & chunk
    chunks = process_markdown_file(md_path, doc_type_override=doc_type)
    
    if not chunks:
        print("❌ No chunks generated!")
        return
    
    # 2. Generate embeddings
    print("\n⏳ Generating embeddings...")
    embeddings = generate_embeddings(chunks)
    print(f"✅ Generated {len(embeddings)} embeddings")
    
    # 3. Upsert to Pinecone
    upsert_chunks(chunks, embeddings)
    
    print(f"\n🎉 Successfully ingested {len(chunks)} chunks from {md_path}")
    return chunks


# Example usage (uncomment to use):
# new_chunks = ingest_new_document("path/to/new_document.md", doc_type="book")
print("✅ Ingestion utility ready. Use ingest_new_document() to add new files.")

---

## 📋 Pipeline Summary

| Step | Component | Status |
|------|-----------|--------|
| 1 | MinerU Markdown Input | ✅ |
| 2 | Structure Parser | ✅ |
| 3 | Document Classifier | ✅ |
| 4 | Heading-aware Book Chunker | ✅ |
| 5 | Q&A-aware Chunker | ✅ |
| 6 | Metadata Layer | ✅ |
| 7 | Embedding Generation | ✅ |
| 8 | Pinecone Vector DB | ✅ |
| 9 | Retrieval | ✅ |
| 10 | Answer Review Agent (Groq) | ✅ |

## 15. Full Exam Assessment — Grade All Questions & Generate Comprehensive Report PDF

Grade an entire multi-question student test in a loop, calculate final scores & analytics, and compile a single multi-page executive assessment PDF with formula images and citations.

In [13]:
def generate_full_exam_pdf(
    student_name: str,
    total_score: float,
    max_score: float,
    all_results: list[dict],
    output_filename: str = "full_exam_review.pdf"
) -> str:
    """
    Generate a comprehensive multi-page assessment report for an entire student test.
    Features:
    - Executive Scorecard Summary Table
    - Question-by-question detailed breakdown
    - Embedded math formula images + strings
    - Exact textbook page citations
    """
    doc = SimpleDocTemplate(
        output_filename,
        pagesize=letter,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )
    
    styles = getSampleStyleSheet()
    story = []
    
    
    pct = (total_score / max_score * 100) if max_score > 0 else 0
    if pct >= 75:
        score_color = colors.HexColor('#10B981') # Green
        grade_desc = "Distinction / Strong Performance"
    elif pct >= 50:
        score_color = colors.HexColor('#F59E0B') # Amber
        grade_desc = "Pass / Satisfactory"
    else:
        score_color = colors.HexColor('#EF4444') # Red
        grade_desc = "Needs Remediation / Study Required"

    title_style = ParagraphStyle(
        'MainTitle',
        parent=styles['Heading1'],
        fontSize=18,
        leading=22,
        textColor=colors.HexColor('#1E3A8A'),
        fontName='Helvetica-Bold'
    )
    subtitle_style = ParagraphStyle(
        'Subtitle',
        parent=styles['Normal'],
        fontSize=9,
        leading=12,
        textColor=colors.HexColor('#4B5563')
    )
    score_box_style = ParagraphStyle(
        'ScoreBox',
        parent=styles['Normal'],
        fontSize=15,
        leading=18,
        alignment=1,
        textColor=colors.white,
        fontName='Helvetica-Bold'
    )
    score_sub_style = ParagraphStyle(
        'ScoreSub',
        parent=styles['Normal'],
        fontSize=8,
        leading=10,
        alignment=1,
        textColor=colors.white
    )

    # 1. Header Banner Table
    header_left = [
        Paragraph("GradeForge Comprehensive Exam Report", title_style),
        Spacer(1, 2),
        Paragraph(f"<b>Student:</b> {student_name} | <b>Date:</b> {datetime.now().strftime('%b %d, %Y')} | <b>Subject:</b> Large Language Models", subtitle_style),
        Paragraph(f"<b>Performance Band:</b> {grade_desc}", subtitle_style)
    ]
    
    header_right = [
        Paragraph(f"TOTAL: {total_score:.1f} / {max_score:.0f}", score_box_style),
        Paragraph(f"Overall Score ({pct:.1f}%)", score_sub_style)
    ]
    
    header_table = Table([[header_left, header_right]], colWidths=[390, 140])
    header_table.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BACKGROUND', (1,0), (1,0), score_color),
        ('ROUNDEDCORNERS', [4, 4, 4, 4]),
        ('TOPPADDING', (1,0), (1,0), 6),
        ('BOTTOMPADDING', (1,0), (1,0), 6),
    ]))
    story.append(header_table)
    story.append(Spacer(1, 10))
    story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#E5E7EB'), spaceBefore=2, spaceAfter=8))

    # 2. Executive Scorecard Summary Table
    sec_style = ParagraphStyle(
        'SectionHeader',
        parent=styles['Heading2'],
        fontSize=11,
        leading=14,
        textColor=colors.HexColor('#1E40AF'),
        fontName='Helvetica-Bold',
        spaceBefore=6,
        spaceAfter=4,
        keepWithNext=True
    )
    th_style = ParagraphStyle(
        'TableHeader',
        parent=styles['Normal'],
        fontSize=8.5,
        leading=11,
        fontName='Helvetica-Bold',
        textColor=colors.HexColor('#1F2937'),
        alignment=1
    )
    td_style = ParagraphStyle(
        'TableCell',
        parent=styles['Normal'],
        fontSize=8.5,
        leading=11,
        textColor=colors.HexColor('#1F2937')
    )
    td_center = ParagraphStyle(
        'TableCellCenter',
        parent=td_style,
        alignment=1
    )

    story.append(Paragraph("<b>📊 Test Scorecard Summary</b>", sec_style))
    summary_data = [[
        Paragraph("Q#", th_style),
        Paragraph("Question Topic", th_style),
        Paragraph("Marks", th_style),
        Paragraph("Status", th_style),
        Paragraph("Primary Study Citation", th_style),
    ]]

    for idx, res in enumerate(all_results):
        score_val = res.get('score', 0)
        q_text = res.get('question', '')[:65] + '...' if len(res.get('question', '')) > 65 else res.get('question', '')
        
        if score_val >= 8:
            status_text = "🟢 Strong"
        elif score_val >= 5:
            status_text = "🟡 Partial"
        else:
            status_text = "🔴 Review"
            
        cites = res.get('citations', '')
        page_match = re.search(r'Page[s]?:\s*([0-9\-]+)', cites)
        page_str = f"p. {page_match.group(1)}" if page_match else "See review"

        summary_data.append([
            Paragraph(f"<b>Q{idx+1}</b>", td_center),
            Paragraph(clean_text_for_xml(_latex_to_plain(q_text)), td_style),
            Paragraph(f"<b>{score_val}/10</b>", td_center),
            Paragraph(status_text, td_center),
            Paragraph(clean_text_for_xml(page_str), td_center),
        ])

    sum_table = Table(summary_data, colWidths=[35, 235, 60, 75, 125])
    sum_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#F3F4F6')),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#E5E7EB')),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ]))
    story.append(sum_table)
    story.append(Spacer(1, 14))
    story.append(PageBreak())

    # 3. Question-by-Question Detailed Section
    box_body_style = ParagraphStyle(
        'BoxBody',
        parent=styles['Normal'],
        fontSize=9,
        leading=13,
        textColor=colors.HexColor('#111827')
    )

    story.append(Paragraph("<b>📝 Detailed Question-by-Question Evaluations</b>", title_style))
    story.append(Spacer(1, 8))

    for idx, res in enumerate(all_results):
        score_val = res.get('score', 0)
        q_hdr = ParagraphStyle(
            f'QHdr_{idx}',
            parent=sec_style,
            fontSize=11.5,
            leading=15,
            textColor=colors.HexColor('#1E3A8A'),
            spaceBefore=10,
            spaceAfter=4,
            keepWithNext=True
        )
        story.append(Paragraph(f"<b>Question {idx+1} Evaluation (Score: {score_val} / 10)</b>", q_hdr))
        story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor('#CBD5E1'), spaceBefore=2, spaceAfter=6))
        
        # Question box
        q_table = Table([[
            Paragraph(f"<b>Question {idx+1}:</b>", sec_style),
            Paragraph(_inline_formula_markup(res.get('question', '')), box_body_style)
        ]], colWidths=[90, 440])
        q_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#F8FAFC')),
            ('ROUNDEDCORNERS', [4, 4, 4, 4]),
            ('TOPPADDING', (0,0), (-1,-1), 6),
            ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ]))
        story.append(q_table)
        story.append(Spacer(1, 5))

        # Student Answer box
        ans_table = Table([[
            Paragraph("<b>Student Answer:</b>", sec_style),
            Paragraph(_inline_formula_markup(res.get('student_answer', '').strip()), box_body_style)
        ]], colWidths=[90, 440])
        ans_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#EFF6FF')),
            ('ROUNDEDCORNERS', [4, 4, 4, 4]),
            ('TOPPADDING', (0,0), (-1,-1), 6),
            ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ]))
        story.append(ans_table)
        story.append(Spacer(1, 8))

        # Citations Box
        cites = res.get('citations', '')
        if cites:
            is_no_kb = "no matching knowledge base" in cites.lower() or cites.strip() == "None"
            box_bg = colors.HexColor('#F1F5F9') if is_no_kb else colors.HexColor('#FEF3C7')
            cite_table = Table([[
                Paragraph("<b>Study Citations:</b>", sec_style),
                Paragraph(clean_text_for_xml(cites).replace('\n', '<br/>'), box_body_style)
            ]], colWidths=[90, 440])
            cite_table.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,-1), box_bg),
                ('ROUNDEDCORNERS', [4, 4, 4, 4]),
                ('TOPPADDING', (0,0), (-1,-1), 6),
                ('BOTTOMPADDING', (0,0), (-1,-1), 6),
                ('VALIGN', (0,0), (-1,-1), 'TOP'),
            ]))
            story.append(cite_table)
            story.append(Spacer(1, 8))

        # Detailed Evaluation with Formula Images
        rev_flowables = parse_markdown_to_flowables(res.get('review', ''), styles)
        story.extend(rev_flowables)
        story.append(Spacer(1, 14))
        if idx < len(all_results) - 1:
            story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#E2E8F0'), spaceBefore=8, spaceAfter=14))

    doc.build(story)
    return output_filename


def grade_single(question: str, student_answer: str, score_threshold: float = SIMILARITY_THRESHOLD) -> dict:
    """Grade a single question and return structured result data with guardrails."""
    ref_results = retrieve(question, namespaces=[NS_ANSWER_KEY], top_k=3, score_threshold=score_threshold)
    reference_answer = "\n\n".join([r["content"] for r in ref_results[:2]]) if ref_results else ""
    
    book_results = retrieve(question, namespaces=[NS_BOOK], top_k=3, score_threshold=score_threshold)
    book_context = _format_book_context_with_pages(book_results[:3]) if book_results else ""
    citations = _format_citations(book_results[:3])
    
    review = review_answer(
        question=question,
        student_answer=student_answer,
        reference_answer=reference_answer,
        book_context=book_context,
        citations=citations,
    )
    
    # Extract numerical score
    score_match = re.search(r'(?:Score|score)[:\s*]+([0-9]+(?:\.[0-9]+)?)\s*[/]\s*10', review, re.IGNORECASE)
    score_val = float(score_match.group(1)) if score_match else 0.0
    
    return {
        "question": question,
        "student_answer": student_answer,
        "score": score_val,
        "review": review,
        "reference_answer": reference_answer,
        "citations": citations,
    }


def grade_full_student_test(
    student_submissions: list[dict],
    student_name: str = "Student",
    output_filename: str = "full_exam_review.pdf"
) -> dict:
    """
    Grade all questions in a student's test submission and generate a complete exam report PDF.
    """
    print("="*60)
    print(f"🎓 GRADEFORGE: Grading Complete Test for {student_name}")
    print(f"   Total Questions: {len(student_submissions)}")
    print("="*60)
    
    all_results = []
    total_score = 0.0
    max_score = len(student_submissions) * 10.0
    
    for idx, item in enumerate(student_submissions):
        q = item.get("question", "")
        a = item.get("student_answer", "")
        print(f"\n⏳ Grading Question {idx + 1} of {len(student_submissions)}: {q[:60]}...")
        
        review_data = grade_single(q, a)
        all_results.append(review_data)
        total_score += review_data["score"]
        print(f"   ✅ Score: {review_data['score']}/10")
        
        # Respect Groq TPM rate limits
        if idx < len(student_submissions) - 1:
            time.sleep(1.5)
            
    pct = (total_score / max_score * 100) if max_score > 0 else 0
    print("\n" + "="*60)
    print(f"📊 FINAL EXAM SCORE: {total_score:.1f} / {max_score:.0f} ({pct:.1f}%)")
    print("="*60)
    
    # Generate the comprehensive PDF report
    print(f"\n📄 Generating Comprehensive Exam Report PDF: {output_filename}...")
    try:
        generate_full_exam_pdf(
            student_name=student_name,
            total_score=total_score,
            max_score=max_score,
            all_results=all_results,
            output_filename=output_filename
        )
        print(f"🎉 Complete Exam Report PDF successfully generated: {output_filename}")
    except Exception as e:
        print(f"⚠️ Error generating full exam PDF: {e}")
        import traceback
        traceback.print_exc()
        
    return {
        "student_name": student_name,
        "total_score": total_score,
        "max_score": max_score,
        "percentage": pct,
        "results": all_results,
        "pdf_path": output_filename,
    }

print("✅ Full Exam Grading & Multi-Page PDF Generation Engine Ready")

✅ Full Exam Grading & Multi-Page PDF Generation Engine Ready


In [ ]:
# ═══════════════════════════════════════════════════════════
# DEMO: Grade a Complete Student Test Submission (Multi-Question)
# ═══════════════════════════════════════════════════════════

student_test_submissions = [
    {
        "question": "what is llm?",
        "student_answer": "it is low long message ,which is use to send message very fast"
    },
    {
        "question": "How is the softmax function applied in attention mechanisms?",
        "student_answer": "The softmax function normalizes attention scores into a probability distribution:"
    },
    {
        "question": "What is tokenization in LLMs and why is it important?",
        "student_answer": "Tokenization breaks input text into tokens so the model can process numbers instead of raw strings. It is critical for vocabulary mapping."
    }
]

# Run full test evaluation and generate consolidated PDF report
full_report = grade_full_student_test(
    student_submissions=student_test_submissions,
    student_name="Nilesh",
    output_filename="full_exam_review.pdf"
)